# 17 — Creator-separated gate calibration on Mac

This notebook runs **without a GPU**. It compares three frozen systems on three
creator videos excluded from every V6 train/validation/test split:

1. V6-B gate + V6-B exercise classifier;
2. V6-C gate + V6-C exercise classifier;
3. V6-C gate + V6-B exercise classifier.

For each gate, the threshold is chosen by the mean binary Macro F1 across
creators, preventing the longest video from dominating. The locked
GainsByBrains evaluation is never downloaded or read.

The calibration archive contains 91 labelled segments from three creators. Its
manifest marks them reviewed, but older annotation notes still contain a draft
warning. This is therefore a **provisional independent calibration set** until
the timelines receive one final spot-check.

Run locally in a Python 3.12 Jupyter environment on the M2 Mac. Colab CPU also
works, but no T4 is needed. Output:
`calorie_v6_gate_calibration_report.zip`.


In [ ]:
%pip install -q torch numpy pandas==2.2.3 scikit-learn matplotlib seaborn


In [ ]:
from pathlib import Path
import json
import shutil
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, recall_score,
)
from torch import nn

SEED = 42
WINDOW = 64
STRIDE = 32
RETIRED_CLASSES = {"plank"}
CALIBRATION_GROUPS = {
    "youtube:oLsonHQJOUc",
    "youtube:c4TomZjzobY",
    "youtube:W2Bm9dfQjbk",
}
np.random.seed(SEED)
torch.manual_seed(SEED)

# CPU is deliberate: inference and threshold sweeps are small and deterministic.
DEVICE = torch.device("cpu")
WORK_ROOT = Path.cwd() / "calorie_local_work"
DOWNLOAD_ROOT = WORK_ROOT / "downloads"
EXTRACT_ROOT = WORK_ROOT / "extracted"
REPORT_ROOT = WORK_ROOT / "calorie_v6_gate_calibration_report"
for path in [DOWNLOAD_ROOT, EXTRACT_ROOT, REPORT_ROOT]:
    path.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE)
print("Work root:", WORK_ROOT.resolve())


## Download compact frozen evidence

Only the two trained reports and one compact landmark archive are downloaded.
No raw video or MediaPipe model is required.


In [ ]:
RAW = "https://raw.githubusercontent.com/bariscelikk1/cALorie/feat/tcn-rebuild/ml/artifacts"
ASSETS = {
    "v6b_report": f"{RAW}/v6b/model/calorie_v6b_augmentation_tcn_report.zip",
    "v6c_report": f"{RAW}/v6c/model/calorie_v6c_kinematic_features_tcn_report.zip",
    "calibration_landmarks": f"{RAW}/v2/landmarks/calorie_youtube_mixed_landmarks_v3.zip",
}


def download(name, url):
    destination = DOWNLOAD_ROOT / Path(url).name
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=300) as response, destination.open("wb") as output:
        shutil.copyfileobj(response, output)
    print(name, "->", destination.name, round(destination.stat().st_size / 1e6, 2), "MB")
    return destination


downloaded = {name: download(name, url) for name, url in ASSETS.items()}
for name, archive_path in downloaded.items():
    destination = EXTRACT_ROOT / name
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir()
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(destination)


## Rebuild independent calibration windows

Each labelled segment becomes one or more 64-frame windows at the same 15 FPS
used for training. Retired plank segments map to `unknown`. No window from
these creators appears in V6-B or V6-C split assignments.


In [ ]:
def resample_sequence(sequence, source_fps, target_fps=15):
    if not source_fps or abs(float(source_fps) - target_fps) < 0.1:
        return sequence
    duration = (len(sequence) - 1) / float(source_fps)
    target_length = max(2, int(round(duration * target_fps)) + 1)
    old_x = np.linspace(0, duration, len(sequence))
    new_x = np.linspace(0, duration, target_length)
    flat = sequence.reshape(len(sequence), -1)
    sampled = np.stack([
        np.interp(new_x, old_x, flat[:, column])
        for column in range(flat.shape[1])
    ], axis=1)
    return sampled.reshape(target_length, 17, 4).astype(np.float32)


landmark_root = EXTRACT_ROOT / "calibration_landmarks"
manifest_path = next(landmark_root.rglob("manifest.csv"))
manifest = pd.read_csv(manifest_path).fillna("")
manifest["npz_path"] = manifest.output_path.map(
    lambda value: str(manifest_path.parent / value)
)
manifest = manifest[
    (manifest.status == "ok")
    & manifest.npz_path.map(lambda value: Path(value).exists())
    & manifest.group_id.isin(CALIBRATION_GROUPS)
].copy()
assert set(manifest.group_id) == CALIBRATION_GROUPS

windows, labels, groups, clip_ids, starts = [], [], [], [], []
for row in manifest.itertuples(index=False):
    with np.load(row.npz_path, allow_pickle=False) as item:
        sequence = item["landmarks"].astype(np.float32)
    sequence = resample_sequence(sequence, getattr(row, "source_fps", 15) or 15)
    clip_starts = list(range(0, max(1, len(sequence) - WINDOW + 1), STRIDE))
    if len(sequence) > WINDOW and clip_starts[-1] != len(sequence) - WINDOW:
        clip_starts.append(len(sequence) - WINDOW)
    for start in clip_starts:
        window = sequence[start:start + WINDOW]
        if len(window) < WINDOW:
            window = np.concatenate([
                window, np.repeat(window[-1:], WINDOW - len(window), axis=0)
            ])
        windows.append(window)
        label = "unknown" if row.canonical_class in RETIRED_CLASSES else row.canonical_class
        labels.append(label)
        groups.append(row.group_id)
        clip_ids.append(row.clip_id)
        starts.append(start)

X = np.asarray(windows, dtype=np.float32)
labels = np.asarray(labels)
groups = np.asarray(groups)
clip_ids = np.asarray(clip_ids)
starts = np.asarray(starts)
y_gate = (labels != "unknown").astype(np.int64)

for version in ["v6b_report", "v6c_report"]:
    split_path = EXTRACT_ROOT / version / "split_assignments.csv"
    split_groups = set(pd.read_csv(split_path).group_id)
    assert not (set(groups) & split_groups), f"Calibration leakage into {version}"

print("Calibration segments:", len(manifest))
print("Calibration windows:", len(X))
display(pd.crosstab(pd.Series(groups, name="creator"), pd.Series(labels, name="label")))


## Load frozen V6-B and V6-C systems

V6-B consumes 34 x/y channels. V6-C consumes 100 channels containing x/y,
velocity, bone vectors, and joint angles. No parameter is trained here.


In [ ]:
BONE_EDGES = [
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12),
    (11, 13), (13, 15), (12, 14), (14, 16),
]
ANGLE_TRIPLETS = [
    (5, 7, 9), (6, 8, 10), (7, 5, 11), (8, 6, 12),
    (5, 11, 13), (6, 12, 14), (11, 13, 15), (12, 14, 16),
]


def features_v6b(sample):
    return sample[:, :, :2].reshape(WINDOW, -1).astype(np.float32)


def features_v6c(sample):
    xy = sample[:, :, :2].astype(np.float32)
    velocity = np.zeros_like(xy)
    velocity[1:] = xy[1:] - xy[:-1]
    bones = np.stack([xy[:, child] - xy[:, parent] for parent, child in BONE_EDGES], axis=1)
    angles = []
    for first, center, last in ANGLE_TRIPLETS:
        a = xy[:, first] - xy[:, center]
        b = xy[:, last] - xy[:, center]
        denominator = np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1)
        cosine = np.sum(a * b, axis=1) / np.maximum(denominator, 1e-6)
        angles.append(np.arccos(np.clip(cosine, -1.0, 1.0)) / np.pi)
    return np.concatenate([
        xy.reshape(WINDOW, -1), velocity.reshape(WINDOW, -1),
        bones.reshape(WINDOW, -1), np.stack(angles, axis=1),
    ], axis=1).astype(np.float32)


class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation, dropout=0.2):
        super().__init__()
        padding = dilation
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
        )
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        return torch.relu(self.net(x) + self.residual(x))


class SmallTCN(nn.Module):
    def __init__(self, input_features, num_classes):
        super().__init__()
        channels = [64, 96, 128, 128]
        blocks, current = [], input_features
        for level, output in enumerate(channels):
            blocks.append(TemporalBlock(current, output, dilation=2 ** level))
            current = output
        self.features = nn.Sequential(*blocks)
        self.classifier = nn.Linear(current, num_classes)

    def forward(self, x):
        return self.classifier(self.features(x).mean(dim=-1))


def load_system(version, feature_function):
    root = EXTRACT_ROOT / f"{version}_report"
    checkpoint_path = next(root.glob("two_stage_tcn_*.pt"))
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    supported = checkpoint["supported_classes"]
    gate = SmallTCN(checkpoint["input_features"], 2).to(DEVICE)
    exercise = SmallTCN(checkpoint["input_features"], len(supported)).to(DEVICE)
    gate.load_state_dict(checkpoint["gate_state_dict"])
    exercise.load_state_dict(checkpoint["exercise_state_dict"])
    gate.eval(); exercise.eval()
    features = np.stack([feature_function(window) for window in X]).transpose(0, 2, 1)
    gate_parts, exercise_parts = [], []
    with torch.no_grad():
        for start in range(0, len(features), 256):
            batch = torch.from_numpy(features[start:start + 256]).to(DEVICE)
            gate_parts.append(gate(batch).softmax(1).cpu().numpy())
            exercise_parts.append(exercise(batch).softmax(1).cpu().numpy())
    return checkpoint, np.concatenate(gate_parts), np.concatenate(exercise_parts)


v6b_checkpoint, v6b_gate, v6b_exercise = load_system("v6b", features_v6b)
v6c_checkpoint, v6c_gate, v6c_exercise = load_system("v6c", features_v6c)
assert v6b_checkpoint["supported_classes"] == v6c_checkpoint["supported_classes"]
SUPPORTED_CLASSES = v6c_checkpoint["supported_classes"]
print("Frozen classes:", SUPPORTED_CLASSES)


## Calibrate each gate with creator-equal scoring

The threshold objective is the mean binary Macro F1 across the three creators.
The sweep also reports global scores and both recalls so a threshold cannot look
good merely by rejecting everything or accepting everything.


In [ ]:
def gate_sweep(name, supported_probability):
    rows = []
    for threshold in np.linspace(0.05, 0.95, 181):
        prediction = (supported_probability >= threshold).astype(np.int64)
        creator_scores = [
            f1_score(y_gate[groups == group], prediction[groups == group], average="macro", zero_division=0)
            for group in sorted(CALIBRATION_GROUPS)
        ]
        rows.append({
            "gate": name,
            "threshold": float(threshold),
            "creator_mean_macro_f1": float(np.mean(creator_scores)),
            "creator_min_macro_f1": float(np.min(creator_scores)),
            "global_macro_f1": float(f1_score(y_gate, prediction, average="macro", zero_division=0)),
            "balanced_accuracy": float(balanced_accuracy_score(y_gate, prediction)),
            "unknown_recall": float(recall_score(y_gate, prediction, pos_label=0, zero_division=0)),
            "supported_recall": float(recall_score(y_gate, prediction, pos_label=1, zero_division=0)),
        })
    frame = pd.DataFrame(rows)
    best = frame.sort_values(
        ["creator_mean_macro_f1", "creator_min_macro_f1", "global_macro_f1"],
        ascending=False,
    ).iloc[0]
    return frame, best


v6b_sweep, v6b_best = gate_sweep("v6b", v6b_gate[:, 1])
v6c_sweep, v6c_best = gate_sweep("v6c", v6c_gate[:, 1])
thresholds = {"v6b": float(v6b_best.threshold), "v6c": float(v6c_best.threshold)}
display(pd.DataFrame([v6b_best, v6c_best]).reset_index(drop=True))


## Compare complete frozen systems and the modular hybrid

The hybrid uses V6-C only for supported-vs-unknown gating and V6-B only for the
accepted exercise label. Selection is based on this independent calibration
evidence, not on the locked long-video test.


In [ ]:
SYSTEMS = {
    "v6b": (v6b_gate[:, 1], v6b_exercise, thresholds["v6b"]),
    "v6c": (v6c_gate[:, 1], v6c_exercise, thresholds["v6c"]),
    "hybrid_v6c_gate_v6b_exercise": (v6c_gate[:, 1], v6b_exercise, thresholds["v6c"]),
}
present_labels = sorted(set(labels))
comparison_rows, prediction_columns, reports = [], {}, {}
for name, (gate_probability, exercise_probability, threshold) in SYSTEMS.items():
    exercise_prediction = np.asarray([
        SUPPORTED_CLASSES[index] for index in exercise_probability.argmax(1)
    ])
    prediction = np.where(gate_probability >= threshold, exercise_prediction, "unknown")
    prediction_columns[name] = prediction
    creator_f1 = [
        f1_score(
            labels[groups == group], prediction[groups == group],
            labels=sorted(set(labels[groups == group])), average="macro", zero_division=0,
        ) for group in sorted(CALIBRATION_GROUPS)
    ]
    comparison_rows.append({
        "system": name,
        "threshold": threshold,
        "accuracy": accuracy_score(labels, prediction),
        "macro_f1_present_classes": f1_score(
            labels, prediction, labels=present_labels, average="macro", zero_division=0
        ),
        "creator_mean_macro_f1": np.mean(creator_f1),
        "creator_min_macro_f1": np.min(creator_f1),
        "unknown_recall": recall_score(labels, prediction, labels=["unknown"], average="macro", zero_division=0),
    })
    reports[name] = classification_report(
        labels, prediction, labels=present_labels, output_dict=True, zero_division=0
    )

comparison = pd.DataFrame(comparison_rows).sort_values(
    ["creator_mean_macro_f1", "creator_min_macro_f1"], ascending=False
).reset_index(drop=True)
display(comparison)
SELECTED_SYSTEM = str(comparison.iloc[0].system)
print("Provisional selected system:", SELECTED_SYSTEM)


## Export the auditable calibration report

The report contains every threshold, per-window predictions, per-system class
reports, and creator-level metrics. No weights are changed.


In [ ]:
sweep = pd.concat([v6b_sweep, v6c_sweep], ignore_index=True)
sweep.to_csv(REPORT_ROOT / "gate_threshold_sweep.csv", index=False)
comparison.to_csv(REPORT_ROOT / "system_comparison.csv", index=False)

window_frame = pd.DataFrame({
    "group_id": groups,
    "clip_id": clip_ids,
    "window_start_frame": starts,
    "true_class": labels,
    "v6b_gate_supported_probability": v6b_gate[:, 1],
    "v6c_gate_supported_probability": v6c_gate[:, 1],
})
for name, prediction in prediction_columns.items():
    window_frame[f"{name}_prediction"] = prediction
window_frame.to_csv(REPORT_ROOT / "calibration_window_predictions.csv", index=False)

for name, report in reports.items():
    (REPORT_ROOT / f"{name}_classification_report.json").write_text(json.dumps(report, indent=2))

summary = {
    "experiment": "creator_separated_gate_calibration",
    "provisional": True,
    "evidence_note": "Manifest says reviewed; historical annotation notes retain a draft warning pending final spot-check.",
    "calibration_groups": sorted(CALIBRATION_GROUPS),
    "calibration_segments": int(len(manifest)),
    "calibration_windows": int(len(X)),
    "present_classes": present_labels,
    "thresholds": thresholds,
    "selected_system": SELECTED_SYSTEM,
    "selection_rule": "creator mean end-to-end Macro F1, then creator minimum Macro F1",
    "locked_test_used": False,
}
(REPORT_ROOT / "calibration_summary.json").write_text(json.dumps(summary, indent=2))

plt.figure(figsize=(9, 5))
for gate_name, group in sweep.groupby("gate"):
    plt.plot(group.threshold, group.creator_mean_macro_f1, label=gate_name)
    plt.axvline(thresholds[gate_name], linestyle="--", alpha=0.6)
plt.xlabel("Gate threshold")
plt.ylabel("Mean creator Macro F1")
plt.title("Independent creator-separated gate calibration")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(REPORT_ROOT / "gate_calibration_curve.png", dpi=180)
plt.show()

selected_prediction = prediction_columns[SELECTED_SYSTEM]
matrix = confusion_matrix(labels, selected_prediction, labels=present_labels)
plt.figure(figsize=(9, 8))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", xticklabels=present_labels, yticklabels=present_labels)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title(f"Calibration confusion matrix — {SELECTED_SYSTEM}")
plt.tight_layout()
plt.savefig(REPORT_ROOT / "selected_system_confusion_matrix.png", dpi=180)
plt.show()

zip_path = shutil.make_archive(str(REPORT_ROOT), "zip", root_dir=REPORT_ROOT)
print("Calibration report:", Path(zip_path).resolve())
print("ZIP size MB:", round(Path(zip_path).stat().st_size / 1024**2, 2))


## Next controlled step

Return `calorie_v6_gate_calibration_report.zip`. After checking creator-level
stability and performing the timeline spot-check, freeze the selected gate,
exercise classifier, and threshold. The next experiment is dense long-video
decoding with more frequent windows, hysteresis, minimum duration, and
short-gap merging.
